# Market/Sentiment Agent Fine-Tuning

This notebook contains the **full fine-tuning code** for the Market + Volatility + News Sentiment Agent.

- Agent file: `agents/market_sentiment_agent.py`
- Data sources: `data/silver/silver_prices_<TICKER>.csv` and `data/silver/silver_news_sentiment.csv`
- Training target: weighted Gold `volatility_risk` + `sentiment_risk`
- Fine-tuning method: LoRA adapter on a causal language model

Run on a GPU environment. CPU training will be very slow.

## 0. Install Dependencies

In [ ]:
# Run this if you see torch/transformers import errors, then RESTART the kernel.
# The _cast_Long error is usually a torch/transformers version mismatch.
#
# %pip uninstall -y torch torchvision torchaudio transformers peft accelerate datasets
# %pip install "torch==2.3.1" "transformers==4.44.2" "datasets==2.21.0" "peft==0.12.0" "accelerate==0.33.0"
#
# After install finishes: Kernel -> Restart Kernel, then rerun from the top.

## 1. Configuration

In [ ]:
import csv
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from agents.market_sentiment_agent import MarketSentimentAgent

BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'

GOLD_PATH = PROJECT_ROOT / 'data/gold/gold_risk_scores_ALL.csv'
TRAIN_JSONL = PROJECT_ROOT / 'data/gold/market_sentiment_finetune_data.jsonl'
OUTPUT_DIR = PROJECT_ROOT / 'models/market_sentiment_lora'

MAX_LENGTH = 2048
EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 2e-4

agent = MarketSentimentAgent(
    silver_dir=str(PROJECT_ROOT / 'data/silver'),
    news_path=str(PROJECT_ROOT / 'data/silver/silver_news_sentiment.csv'),
)

print('Project root:', PROJECT_ROOT)
print('Gold file:', GOLD_PATH)
print('Train JSONL:', TRAIN_JSONL)
print('Output dir:', OUTPUT_DIR)

## 2. Create Market/Sentiment Fine-Tuning JSONL

In [ ]:
def label_from_score(score):
    score = float(score)
    if score <= 3.5:
        return 'LOW'
    if score <= 6.0:
        return 'MODERATE'
    return 'HIGH'

def load_gold_rows(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

gold_rows = load_gold_rows(GOLD_PATH)
examples = []

for row in gold_rows:
    ticker = row['ticker'].upper()
    try:
        result = agent.analyze(ticker)
    except Exception:
        continue

    volatility_score = float(row.get('volatility_risk') or row['composite_risk'])
    sentiment_score = float(row.get('sentiment_risk') or 5.0)
    target_score = round(volatility_score * 0.70 + sentiment_score * 0.30, 2)
    target_label = label_from_score(target_score)

    user_payload = {
        'ticker': ticker,
        'task': 'Assess market, volatility, and news sentiment risk.',
        'features': result.features,
        'evidence': result.evidence,
    }
    assistant_payload = {
        'market_sentiment_risk_score': target_score,
        'market_sentiment_risk_label': target_label,
        'explanation': 'Assess volatility, beta, drawdown, price trend, article volume, sentiment direction, and sentiment uncertainty.'
    }

    examples.append({
        'messages': [
            {'role': 'system', 'content': 'You are a market volatility and financial news sentiment risk analyst. Return strict JSON.'},
            {'role': 'user', 'content': json.dumps(user_payload)},
            {'role': 'assistant', 'content': json.dumps(assistant_payload)},
        ]
    })

TRAIN_JSONL.parent.mkdir(parents=True, exist_ok=True)
with open(TRAIN_JSONL, 'w', encoding='utf-8') as f:
    for ex in examples:
        f.write(json.dumps(ex) + '\n')

print('Training examples:', len(examples))
print('Saved:', TRAIN_JSONL)
print(json.dumps(examples[0], indent=2)[:1500])

## 3. Load JSONL Messages

In [ ]:
def load_messages(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line)['messages'])
    return rows

messages = load_messages(TRAIN_JSONL)
print('Loaded examples:', len(messages))

## 4. Tokenizer And Dataset

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_messages(chat_messages):
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        return tokenizer.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=False)
    return '\n\n'.join(f"{m['role'].upper()}: {m['content']}" for m in chat_messages) + tokenizer.eos_token

texts = [format_messages(row) for row in messages]
dataset = Dataset.from_dict({'text': texts})

def tokenize(batch):
    tokenized = tokenizer(batch['text'], max_length=MAX_LENGTH, truncation=True, padding='max_length')
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=['text'])
print(tokenized_dataset)

## 5. Load Base Model And Attach LoRA

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map='auto' if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Train

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    logging_steps=10,
    save_strategy='epoch',
    report_to='none',
    fp16=torch.cuda.is_available() and dtype == torch.float16,
    bf16=torch.cuda.is_available() and dtype == torch.bfloat16,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

## 7. Save Adapter

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved LoRA adapter to:', OUTPUT_DIR)

## 8. Test Inference

In [ ]:
test_prompt = messages[0][1]['content']
test_messages = [
    {'role': 'system', 'content': 'You are a market volatility and financial news sentiment risk analyst. Return strict JSON.'},
    {'role': 'user', 'content': test_prompt},
]

if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
    prompt_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
else:
    prompt_text = f"SYSTEM: {test_messages[0]['content']}\n\nUSER: {test_messages[1]['content']}\n\nASSISTANT:"

inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))